# Bulk RNA-seq Preprocessing

Convert bulk RNA-seq data (`expression.h5`, `gene_list.txt`, `metadata.csv`) into the
CancerFoundation pipeline format (`vocab.json`, `obs.parquet`, `mapping.json`, `mem.map/`).

In [9]:
# Inputs and general settings
from pathlib import Path

# --- INPUT: bulk data files (in the same directory) ---
# BULK_DIR = Path("/cluster/work/boeva/bulkFM/data/processed/archs4")
BULK_DIR = Path("tutorials/archs4_test")
EXPRESSION_H5 = BULK_DIR / "expression.h5"
GENE_LIST_TXT = BULK_DIR / "gene_list.txt"
METADATA_CSV = BULK_DIR / "metadata.csv"

# --- OUTPUT: pipeline-ready processed data ---
OUTPUT_DIR = BULK_DIR / "h5ads"

# Optional: path to an existing vocab.json to reuse (set to None to generate from gene_list.txt)
EXISTING_VOCAB_PATH = None

# Metadata columns to include in obs.parquet (must exist in metadata.csv)
OBS_COLUMNS = ["tissue", "series_id", "sample_id", "total_counts"]

# How many samples per h5ad chunk (controls memory during conversion)
CHUNK_SIZE = 500

In [3]:
## Imports
import sys
sys.path.insert(0, "./")

import h5py
import json
import os
import shutil
import numpy as np
import pandas as pd
import anndata as ad
import scipy.sparse as sp
from bionemo.scdl.io.single_cell_collection import SingleCellCollection

from cancerfoundation.data.dataset import DatasetDir

GENE_ID = "_cf_gene_id"
CLS_TOKEN = "<cls>"
PAD_TOKEN = "<pad>"

## 1. Load bulk data

In [4]:
# Load gene list
with open(GENE_LIST_TXT) as f:
    gene_names = [line.strip() for line in f if line.strip()]

print(f"Genes: {len(gene_names)}")

# Load metadata
metadata = pd.read_csv(METADATA_CSV)
print(f"Samples in metadata: {len(metadata)}")
print(f"Columns: {list(metadata.columns)}")
metadata.head()

Genes: 67186
Samples in metadata: 1000
Columns: ['sample_id', 'geo_accession', 'tissue', 'series_id', 'total_counts', 'n_detected_genes']


,sample_id,geo_accession,tissue,series_id,total_counts,n_detected_genes
0,sample_000000,GSM_SYNTH_000000,unknown,GSE_SYNTH_000000,70714480.0,32704
1,sample_000001,GSM_SYNTH_000001,unknown,GSE_SYNTH_000000,58300224.0,32070
2,sample_000002,GSM_SYNTH_000002,unknown,GSE_SYNTH_000000,65226924.0,32713
3,sample_000003,GSM_SYNTH_000003,unknown,GSE_SYNTH_000000,72034816.0,33255
4,sample_000004,GSM_SYNTH_000004,unknown,GSE_SYNTH_000000,66846772.0,32888


In [5]:
# Inspect expression matrix shape without loading it all
with h5py.File(EXPRESSION_H5, "r") as f:
    print("Datasets in HDF5:", list(f.keys()))
    # Adapt the dataset key if needed (common keys: "expression", "data/expression", "X")
    for key in f.keys():
        ds = f[key]
        if hasattr(ds, "shape"):
            print(f"  {key}: shape={ds.shape}, dtype={ds.dtype}")

    # Determine the expression dataset key
    expr_key = "expression"  # adjust if your file uses a different key
    n_samples, n_genes_h5 = f[expr_key].shape
    print(f"\nExpression matrix: {n_samples} samples × {n_genes_h5} genes")
    assert n_genes_h5 == len(gene_names), (
        f"Gene count mismatch: h5 has {n_genes_h5}, gene_list.txt has {len(gene_names)}"
    )

Datasets in HDF5: ['expression']
  expression: shape=(1000, 67186), dtype=float32

Expression matrix: 1000 samples × 67186 genes


## 2. Convert expression matrix to chunked h5ad files

Each chunk becomes one h5ad file with:
- `.X`: sparse expression matrix (samples × genes)
- `.var_names`: gene symbols
- `.var["_cf_gene_id"]`: vocab indices per gene
- `.obs`: metadata columns from `metadata.csv`

These h5ad files are an intermediate step consumed by `SingleCellCollection`.

In [10]:
h5ad_dir = OUTPUT_DIR
h5ad_dir.mkdir(parents=True, exist_ok=True)

# Map gene names to vocab IDs; drop genes not in vocab
gene_ids = np.array([vocab.get(g, -1) for g in gene_names])
valid_mask = gene_ids >= 0
valid_gene_names = [g for g, v in zip(gene_names, valid_mask) if v]
valid_gene_ids = gene_ids[valid_mask]
valid_col_indices = np.where(valid_mask)[0]

print(f"Genes in vocab: {len(valid_gene_names)}/{len(gene_names)}")

# Use sample_id as index if available, else generate one
if "sample_id" in metadata.columns:
    metadata = metadata.set_index("sample_id", drop=False)

with h5py.File(EXPRESSION_H5, "r") as f:
    expr_ds = f[expr_key]
    n_total = expr_ds.shape[0]
    n_chunks = (n_total + CHUNK_SIZE - 1) // CHUNK_SIZE

    for chunk_idx in range(n_chunks):
        start = chunk_idx * CHUNK_SIZE
        end = min(start + CHUNK_SIZE, n_total)

        # Read chunk and select valid genes
        X_dense = expr_ds[start:end, :][:, valid_col_indices]
        X_sparse = sp.csr_matrix(X_dense.astype(np.float32))

        obs_chunk = metadata.iloc[start:end][OBS_COLUMNS].copy()
        obs_chunk.index = obs_chunk.index.astype(str)

        var = pd.DataFrame({GENE_ID: valid_gene_ids}, index=valid_gene_names)
        var[GENE_ID] = var[GENE_ID].astype(int)

        adata = ad.AnnData(X=X_sparse, obs=obs_chunk, var=var)

        fname = f"bulk_chunk_{chunk_idx:04d}.h5ad"
        adata.write_h5ad(h5ad_dir / fname)
        print(f"  [{chunk_idx+1}/{n_chunks}] wrote {fname}  ({end - start} samples)")

print(f"\nCreated {n_chunks} h5ad files in {h5ad_dir}")

Genes in vocab: 67186/67186
  [1/2] wrote bulk_chunk_0000.h5ad  (500 samples)
  [2/2] wrote bulk_chunk_0001.h5ad  (500 samples)

Created 2 h5ad files in tutorials/archs4_test/h5ads


## Synthetic data for testing

In [1]:
from pathlib import Path
import h5py
import numpy as np
import pandas as pd

# ===== User settings =====
input_h5 = Path("/cluster/work/boeva/bulkFM/data/processed/archs4/expression.h5")          # path to your full H5 file
output_dir = Path("tutorials/archs4_test")     # where to write the test files
dataset_name = "expression"               # HDF5 dataset name
n_samples = 1000                          # take only the first n samples
gene_prefix = "GENE"
sample_prefix = "sample"

output_dir.mkdir(parents=True, exist_ok=True)

subset_h5 = output_dir / "expression.h5"
gene_list_txt = output_dir / "gene_list.txt"
metadata_csv = output_dir / "metadata.csv"

with h5py.File(input_h5, "r") as f_in:
    ds = f_in[dataset_name]
    total_samples, n_genes = ds.shape
    n = min(n_samples, total_samples)

    print(f"Input shape: {total_samples} samples x {n_genes} genes")
    print(f"Taking subset: first {n} samples")

    # Load only the first n rows for testing
    X = ds[:n, :]

# Write subset H5
with h5py.File(subset_h5, "w") as f_out:
    f_out.create_dataset(
        dataset_name,
        data=X,
        compression="gzip",
        compression_opts=4,
        chunks=True,
    )

# Synthetic gene list
genes = [f"{gene_prefix}_{i:06d}" for i in range(n_genes)]
with open(gene_list_txt, "w", encoding="utf-8") as f:
    for g in genes:
        f.write(g + "\n")

# Synthetic metadata, but with real counts derived from the subset matrix
total_counts = X.sum(axis=1)
n_detected_genes = (X > 0).sum(axis=1)

metadata = pd.DataFrame({
    "sample_id": [f"{sample_prefix}_{i:06d}" for i in range(n)],
    "geo_accession": [f"GSM_SYNTH_{i:06d}" for i in range(n)],
    "tissue": ["unknown"] * n,
    "series_id": [f"GSE_SYNTH_{i // 100:06d}" for i in range(n)],
    "total_counts": total_counts.astype(float),
    "n_detected_genes": n_detected_genes.astype(int),
})

metadata.to_csv(metadata_csv, index=False)

print(f"\nWrote files to: {output_dir.resolve()}")
print(f"  - {subset_h5.name}")
print(f"  - {gene_list_txt.name}")
print(f"  - {metadata_csv.name}")

print("\nMetadata preview:")
display(metadata.head())

print("\nGene list preview:")
print(genes[:5])

Input shape: 617413 samples x 67186 genes
Taking subset: first 1000 samples

Wrote files to: /cluster/work/boeva/rquiles/CancerFoundation/tutorials/archs4_test
  - expression.h5
  - gene_list.txt
  - metadata.csv

Metadata preview:


,sample_id,geo_accession,tissue,series_id,total_counts,n_detected_genes
0,sample_000000,GSM_SYNTH_000000,unknown,GSE_SYNTH_000000,70714480.0,32704
1,sample_000001,GSM_SYNTH_000001,unknown,GSE_SYNTH_000000,58300224.0,32070
2,sample_000002,GSM_SYNTH_000002,unknown,GSE_SYNTH_000000,65226924.0,32713
3,sample_000003,GSM_SYNTH_000003,unknown,GSE_SYNTH_000000,72034816.0,33255
4,sample_000004,GSM_SYNTH_000004,unknown,GSE_SYNTH_000000,66846772.0,32888



Gene list preview:
['GENE_000000', 'GENE_000001', 'GENE_000002', 'GENE_000003', 'GENE_000004']
